# Computational Workflows for biomedical data

Welcome to the course Computational Workflows for Biomedical Data. Over the next two weeks, you will learn how to leverage nf-core pipelines to analyze biomedical data and gain hands-on experience in creating your own pipelines, with a strong emphasis on Nextflow and nf-core.

Course Structure:

- Week 1: You will use a variety of nf-core pipelines to analyze a publicly available biomedical study.
- Week 2: We will shift focus to learning the basics of Nextflow, enabling you to design and implement your own computational workflows.<br>
- Final Project: The last couple of days, you will apply your knowledge to create a custom pipeline for analyzing biomedical data using Nextflow and the nf-core template.

## Basics

If you have not installed all required software, please do so now asap!


If you already installed all software, please go on and start answering the questions in this notebook. If you have any questions, don't hesitate to approach us.

1. What is nf-core?

It is a global community collaborating to build open-source Nextflow components and pipelines. Nextflow is a powerful and flexible workflow language.

2. How many pipelines are there currently in nf-core?

157 (101 released, 43 under development, 13 archived)

3. Are there any non-bioinformatic pipelines in nf-core?

Yes

4. Let's go back a couple of steps. What is a pipeline and what do we use it for?

A pipeline is a series of computational steps chained together so that the output of one step automatically becomes the input of the next, letting data move from raw input to final result without manual intervention at each stage.

5. Why do you think nf-core adheres to strict guidelines?

Reproducibility, quality control, portability, maintainability, documentation

6. What are the main features of nf-core pipelines?

- Extensive documentation covering installation, usage and description of output files
- nf-core pipelines use GitHub releases to tag stable versions of the code and software
- All nf-core code is licensed under the MIT license and is available on GitHub
- Every time a change is made to the pipeline code, nf-core pipelines use continuous-integration testing to ensure that nothing has broken
- Pipelines are ultra-portable and run on most laptops, HPCs and clouds
- Pipeline dependencies are automatically downloaded and handled using Docker, Singularity, Conda or others

## Let's start using the pipelines

1. Find the nf-core pipeline used to measure differential abundance of genes

https://nf-co.re/differentialabundance

In [1]:
# run the pipeline in a cell 
# to run bash in jupyter notebooks, simply use ! before the command
# e.g.

!pwd


# For the tasks in the first week, please use the command line to run your commands and simply paste the commands you used in the respective cells!


/home/laurens/repositories/bioinfo/projects/cwmd/day_01


In [ ]:
# run the pipeline in the test profile using docker containers
# make sure to specify the version you want to use (use the latest one)


!nextflow run nf-core/differentialabundance -r 2.0.0 -profile test,docker -c local.config --outdir results
# approx 20 min run time

In [ ]:
# repeat the run 

!nextflow run nf-core/differentialabundance -r 2.0.0 -profile test,docker -c local.config --outdir results
# approx 7.5 min run time

What did change?

The run got a new random name (nauseous_waddington instead of shrivelled_jennings). All 25 tasks ran again from scratch and wrote new hash folders in work/. The results in results/ were overwritten, and pipeline_info/ gained a second set of reports with a new timestamp. It was faster (about 7 minutes) only because the Docker images were already downloaded. No computation was reused.

In [ ]:
# now set -resume to the command. What did change?

!nextflow run nf-core/differentialabundance -r 2.0.0 -profile test,docker -c local.config --outdir results -resume
# aaprox 0.5 min run time

Check out the current directory. Next to the outdir you specified, what else has changed?

- work/: one hash subfolder per task with .command.sh, .command.run, .command.log, .exitcode and the intermediate files
- .nextflow/: the cache database and history for every run, which nextflow log reads
- .nextflow.log: the log of the latest run. Older logs are rotated to .nextflow.log.1, .2 and so on
- results/pipeline_info/: execution report, timeline, trace, DAG and params for every run

In [ ]:
# delete the work directory and run the pipeline again using -resume. What did change?

!nextflow run nf-core/differentialabundance -r 2.0.0 -profile test,docker -c local.config --outdir results -resume
# approx 7.5 min run time

What changed?

Almost everything ran again: 24 tasks executed and 1 was cached, taking about 5 minutes. -resume needs both the cache records in .nextflow/ and the actual files in work/. Once work/ is gone, those records point to nothing, so the tasks have to run again. The one cached task, GUNZIP_GTF, is only there because an interrupted run at 12:45 had already recreated it. In a clean case, nothing would be cached.

## Lets look at the results

### What is differential abundance analysis?

It's a statistical test, per feature (genes here, but also proteins or metabolites), of whether that feature's abundance differs between conditions. It accounts for differences in sequencing depth and for variation between replicates. For RNA-seq counts, the pipeline uses DESeq2 by default. DESeq2 models the counts with a negative binomial distribution and estimates a log2 fold change and a p-value for each gene. The p-values are then corrected for multiple testing (padj, Benjamini-Hochberg). The results can be passed on to gene set enrichment (GSEA) to see which pathways shift.

The test data comes from study SRP254919. It compares 3 mouse samples with the mutant human ND6 gene (hND6) against 3 mCherry controls, using the top 1000 genes.

Give the most important plots from the report:

![PCA](figures/pca2d.png)

- PC1 explains 45% and PC2 22% of the variance.
- The groups only separate a little, along PC2: the mCherry controls lie higher, the hND6 samples lower.
- PC1, the biggest source of variation, does not follow treatment. SRX8042383 (mCherry) and SRX8042385 (hND6) sit at opposite ends of it.
- Meaning: most of the variation in this data comes from differences between individual samples, not from treatment.

![Volcano](figures/volcano.png)

- Almost all of the roughly 980 genes are grey, clustered near log2FC 0 with padj around 1.
- Only 5 genes pass the cutoffs (padj < 0.05, |log2FC| > 2), and all of them are higher in hND6
- Meaning: the only clear signal is antibody production, i.e. B cells or plasma cells.


# Interesting finds

The hND6 mice show a strong, consistent rise in immunoglobulin genes (Ighg2b, Igkc, Igha, Jchain, Pou2af1), pointing to B cell or plasma cell activity. Otherwise the transcriptome barely differs from the mCherry controls and the samples cluster by individual rather than by treatment.